# dim_player Backfill Notebook

**Issue #165** — One-time backfill to populate `dim_player` from all 32 NHL team rosters.

Fetches all 32 NHL team rosters for season `20252026` via
`GET https://api-web.nhle.com/v1/roster/{team}/20252026` and upserts each player
directly into the Flask app's SQLite database — no Flask app context required.

## Run instructions

```bash
pip install jupyter pandas httpx sqlalchemy pytz
jupyter notebook nhl-dashboard/notebooks/dim_player_backfill.ipynb
```

Run all cells top-to-bottom. Section 3 calls the live NHL API for all 32 teams
— expect ~5–10 s total.

## Notebook structure

| Section | Content |
|---|---|
| Setup | Imports, SQLAlchemy engine, NHL API base URL, 32-team list |
| Section 1 — Single team sample | Fetch one team roster, print response shape and field inventory |
| Section 2 — Upsert function | Define `upsert_player(session, player_dict)` using `session.merge()` |
| Section 3 — Batch fetch (all 32 teams) | Loop all teams, fetch rosters, upsert each player, print progress |
| Section 4 — Verification | Query row count, position breakdown, and sample rows |

## Setup

Imports, SQLAlchemy connection to the Flask app's SQLite database, NHL API base URL,
and the complete list of all 32 NHL team abbreviations.

The engine connects directly to `../backend/nhl_dashboard.db` — no Flask app context
is required. `DimPlayer` is imported from the backend models to keep column definitions
in sync with the live schema.

In [ ]:
import sys
import time
from datetime import datetime
from pathlib import Path
from pprint import pprint

import httpx
import pandas as pd
import pytz
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker

# Add backend to path so we can import models without Flask app context
BACKEND_DIR = Path("../backend").resolve()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

NHL_BASE = "https://api-web.nhle.com/v1"
SEASON   = "20252026"
DB_PATH  = BACKEND_DIR / "nhl_dashboard.db"

# All 32 NHL team abbreviations
NHL_TEAMS = [
    "ANA", "BOS", "BUF", "CGY", "CAR", "CHI", "COL", "CBJ",
    "DAL", "DET", "EDM", "FLA", "LAK", "MIN", "MTL", "NSH",
    "NJD", "NYI", "NYR", "OTT", "PHI", "PIT", "SJS", "SEA",
    "STL", "TBL", "TOR", "UTA", "VAN", "VGK", "WSH", "WPG",
]

SAMPLE_TEAM = "TOR"

# Connect directly to the Flask app's SQLite DB — no app context needed
engine  = create_engine(f"sqlite:///{DB_PATH}", echo=False)
Session = sessionmaker(bind=engine)

ET = pytz.timezone("US/Eastern")


def now_eastern() -> datetime:
    """Return the current datetime in US/Eastern (naive UTC-offset stripped)."""
    return datetime.now(ET).replace(tzinfo=None)


print(f"Database  : {DB_PATH}")
print(f"DB exists : {DB_PATH.exists()}")
print(f"Teams     : {len(NHL_TEAMS)}")
print(f"Season    : {SEASON}")

## Section 1 — Single team sample

Fetches one team roster from `GET /v1/roster/{team}/{season}` and prints the raw
response shape and a field inventory for the first player in each position group.

Use this cell to verify the API response looks as expected before running the full
batch fetch in Section 3.

In [ ]:
resp = httpx.get(f"{NHL_BASE}/roster/{SAMPLE_TEAM}/{SEASON}", timeout=15)
resp.raise_for_status()
sample_roster = resp.json()

print(f"Top-level keys : {list(sample_roster.keys())}")
print()

for group in ("forwards", "defensemen", "goalies"):
    players = sample_roster.get(group, [])
    print(f"{group}: {len(players)} players")
    if players:
        p0 = players[0]
        print(f"  Player keys : {list(p0.keys())}")
        first = p0.get("firstName", {})
        last  = p0.get("lastName", {})
        fname = first.get("default", "?") if isinstance(first, dict) else first
        lname = last.get("default", "?") if isinstance(last, dict) else last
        print(f"  Sample name : {fname} {lname} — #{p0.get('sweaterNumber')} {p0.get('positionCode')}")
    print()

## Section 2 — Upsert function

Defines `upsert_player(session, player_dict)` which:

1. Extracts all `dim_player` columns from the roster API player dict
2. Constructs a `DimPlayer` instance
3. Calls `session.merge()` to upsert by `player_id` (INSERT on first run, UPDATE on re-runs)
4. Sets `updated_at` to the current Eastern time

Helper `_default()` extracts `.default` from localised NHL name dicts.

In [ ]:
# Import the SQLAlchemy model — keeps column definitions in sync with live schema
from models import DimPlayer


def _default(val) -> str | None:
    """Extract .default from a localised NHL name dict, or return the value as-is."""
    return val.get("default") if isinstance(val, dict) else val


def upsert_player(session, player_dict: dict) -> None:
    """Upsert one player row into dim_player from a roster API player dict.

    Uses session.merge() so the call is idempotent: INSERT on first run,
    UPDATE on subsequent runs. sweater_number is always overwritten because
    jersey numbers change between seasons.

    Args:
        session: SQLAlchemy Session bound to the Flask app's SQLite database.
        player_dict: Raw player object from /v1/roster/{team}/{season} response.
    """
    player = DimPlayer(
        player_id        = player_dict.get("id"),
        first_name       = _default(player_dict.get("firstName")),
        last_name        = _default(player_dict.get("lastName")),
        sweater_number   = player_dict.get("sweaterNumber"),
        position         = player_dict.get("positionCode"),
        shoots_catches   = player_dict.get("shootsCatches"),
        height_in_inches = player_dict.get("heightInInches"),
        weight_in_pounds = player_dict.get("weightInPounds"),
        birth_date       = player_dict.get("birthDate"),
        birth_country    = player_dict.get("birthCountry"),
        headshot_url     = player_dict.get("headshot"),
        updated_at       = now_eastern(),
    )
    session.merge(player)


print("upsert_player() defined")
print("Columns populated:")
for col in ["player_id", "first_name", "last_name", "sweater_number", "position",
            "shoots_catches", "height_in_inches", "weight_in_pounds",
            "birth_date", "birth_country", "headshot_url", "updated_at"]:
    print(f"  {col}")

## Section 3 — Batch fetch (all 32 teams)

Loops over all 32 NHL teams, fetches each roster from
`GET /v1/roster/{team}/20252026`, and upserts every player into `dim_player`.

- Rate-limited at 50 ms between requests (same pattern as `player_dim_explorer.ipynb`)
- HTTP errors and non-200 responses are caught and logged; the loop continues
- Progress is printed per team
- All upserts are committed in a single transaction after the loop

In [ ]:
failed_teams   = []
upserted_total = 0

with Session() as session:
    for team in NHL_TEAMS:
        try:
            r = httpx.get(f"{NHL_BASE}/roster/{team}/{SEASON}", timeout=15)
            if r.status_code != 200:
                reason = f"HTTP {r.status_code}"
                failed_teams.append((team, reason))
                print(f"  SKIP {team}: {reason}")
                continue

            data     = r.json()
            players  = (
                data.get("forwards",   []) +
                data.get("defensemen", []) +
                data.get("goalies",    [])
            )

            for p in players:
                upsert_player(session, p)

            upserted_total += len(players)
            print(f"  OK {team}: {len(players)} players upserted")

        except Exception as exc:
            failed_teams.append((team, str(exc)))
            print(f"  ERROR {team}: {exc}")

        time.sleep(0.05)  # polite rate-limiting — 50 ms between requests

    session.commit()

print()
print(f"Batch complete — players upserted : {upserted_total}")
print(f"Failed teams                      : {len(failed_teams)}")
if failed_teams:
    for t, reason in failed_teams:
        print(f"  {t}: {reason}")

## Section 4 — Verification

Queries `dim_player` to confirm the backfill succeeded:

1. **Total row count** — expect ~700–900 active NHL players
2. **Position breakdown** — distribution across C, L, R, D, G
3. **Sample of 10 rows** — spot-check names, sweater numbers, and `updated_at`

In [ ]:
with engine.connect() as conn:
    # Row count
    row_count = conn.execute(text("SELECT COUNT(*) FROM dim_player")).scalar()

    # Position breakdown
    pos_rows = conn.execute(
        text("SELECT position, COUNT(*) AS cnt FROM dim_player GROUP BY position ORDER BY cnt DESC")
    ).fetchall()

    # Sample rows
    sample_rows = conn.execute(
        text(
            "SELECT player_id, first_name, last_name, sweater_number, position, "
            "birth_country, updated_at FROM dim_player LIMIT 10"
        )
    ).fetchall()

print(f"Total rows in dim_player: {row_count}")
print()
print("Position breakdown:")
for pos, cnt in pos_rows:
    print(f"  {pos or 'NULL':4s}: {cnt}")
print()

df_sample = pd.DataFrame(
    sample_rows,
    columns=["player_id", "first_name", "last_name", "sweater_number",
             "position", "birth_country", "updated_at"],
)
print(f"Sample of {len(df_sample)} rows:")
display(df_sample)